In [4]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print(model.forward.__annotations__)

print(model.config.output_attentions)
print(model.config.output_hidden_states)
print(model.config.return_dict)
outputs = model(output_attentions=True, output_hidden_states=True)
print(type(outputs))

{'input_features': torch.FloatTensor | None, 'attention_mask': torch.LongTensor | None, 'decoder_input_ids': torch.LongTensor | None, 'decoder_attention_mask': torch.LongTensor | None, 'head_mask': torch.Tensor | None, 'decoder_head_mask': torch.Tensor | None, 'cross_attn_head_mask': torch.Tensor | None, 'encoder_outputs': tuple[tuple[torch.FloatTensor]] | None, 'past_key_values': transformers.cache_utils.Cache | None, 'decoder_inputs_embeds': tuple[torch.FloatTensor] | None, 'decoder_position_ids': tuple[torch.LongTensor] | None, 'labels': torch.LongTensor | None, 'use_cache': bool | None, 'output_attentions': bool | None, 'output_hidden_states': bool | None, 'return_dict': bool | None, 'cache_position': torch.LongTensor | None, 'return': tuple[torch.Tensor] | transformers.modeling_outputs.Seq2SeqLMOutput}
False
False
True


AttributeError: 'NoneType' object has no attribute 'shape'

In [1]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio
from src.contributions import ModelWrapper
import torch
from pyaml_env import parse_config
config = parse_config("./src/config.yaml")


processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

print(config['models'][model.config.model_type])
print(model.config.num_attention_heads)
print(model.config.hidden_size / model.config.num_attention_heads)
     
model_wrapped = ModelWrapper(model)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model.to(device)
model.model.decoder

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

{'layer': 'model.model.decoder.layers', 'ln1': 'self_attn_layer_norm', 'ln2': 'final_layer_norm', 'values': 'self_attn.v_proj', 'dense': 'self_attn.out_proj', 'lnf': 'model.model.decoder.final_layer_norm', 'fc1': 'fc1', 'fc2': 'fc1', 'unembed': 'proj_out', 'pre_layer_norm': 'True'}
12
64.0
cuda


WhisperDecoder(
  (embed_tokens): Embedding(51865, 768, padding_idx=50257)
  (embed_positions): WhisperPositionalEmbedding(448, 768)
  (layers): ModuleList(
    (0-11): 12 x WhisperDecoderLayer(
      (self_attn): WhisperAttention(
        (k_proj): Linear(in_features=768, out_features=768, bias=False)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (activation_fn): GELUActivation()
      (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (encoder_attn): WhisperAttention(
        (k_proj): Linear(in_features=768, out_features=768, bias=False)
        (v_proj): Linear(in_features=768, out_features=768, bias=True)
        (q_proj): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (encoder

In [2]:
ds = load_dataset("Isma/librispeech_tiny")
ds = ds["10mn"]

def prepare_batch(batch):
    print(batch["audio"]["array"])
    waveform = batch["audio"]["array"]
    sr = batch["audio"]["sampling_rate"]
    batch["input_features"] = processor(
        waveform,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features[0]
    return batch

dataset = ds.map(prepare_batch)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

[0.02813721 0.02984619 0.01861572 ... 0.03112793 0.03240967 0.02627563]
[-3.0517578e-05  6.1035156e-05  9.1552734e-05 ...  0.0000000e+00
  0.0000000e+00  3.0517578e-05]
[2.1362305e-04 1.8310547e-04 1.8310547e-04 ... 9.1552734e-05 1.2207031e-04
 1.2207031e-04]
[-0.00408936 -0.00192261 -0.00094604 ... -0.01248169 -0.01254272
 -0.0083313 ]
[ 2.4414062e-04 -3.0517578e-05 -5.1879883e-04 ... -1.6784668e-03
 -3.2348633e-03 -2.6855469e-03]
[-1.0681152e-03 -5.1879883e-04  3.0517578e-05 ...  2.2277832e-03
  2.1972656e-03  2.4108887e-03]
[ 0.00405884  0.0020752   0.00335693 ...  0.00128174 -0.00027466
 -0.00164795]
[-1.2817383e-03 -7.3242188e-04  3.3569336e-04 ... -9.7656250e-04
  6.1035156e-05  3.6621094e-04]
[0. 0. 0. ... 0. 0. 0.]
[ 0.00015259 -0.00067139 -0.00042725 ... -0.0017395   0.0057373
  0.00753784]
[ 1.2207031e-04 -6.1035156e-05 -4.8828125e-04 ... -2.3498535e-03
 -2.5634766e-03 -3.2348633e-03]
[-0.00024414  0.00119019  0.00064087 ...  0.00079346  0.00064087
  0.00036621]
[ 0.00112915 

In [10]:
model.model.decoder.layers

ModuleList(
  (0-11): 12 x WhisperDecoderLayer(
    (self_attn): WhisperAttention(
      (k_proj): Linear(in_features=768, out_features=768, bias=False)
      (v_proj): Linear(in_features=768, out_features=768, bias=True)
      (q_proj): Linear(in_features=768, out_features=768, bias=True)
      (out_proj): Linear(in_features=768, out_features=768, bias=True)
    )
    (activation_fn): GELUActivation()
    (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder_attn): WhisperAttention(
      (k_proj): Linear(in_features=768, out_features=768, bias=False)
      (v_proj): Linear(in_features=768, out_features=768, bias=True)
      (q_proj): Linear(in_features=768, out_features=768, bias=True)
      (out_proj): Linear(in_features=768, out_features=768, bias=True)
    )
    (encoder_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (fc1): Linear(in_features=768, out_features=3072, bias=True)
    (fc2): Linear(in_features=3072, out

In [3]:
transcriptions = []
for sample in dataset:
    input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

    generated_ids = model.generate(input_features, language="en", task="translate")
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    transcriptions.append(text)

for i, t in enumerate(transcriptions):
    print(f"Sample {i}: {t}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


KeyboardInterrupt: 